# Software Verification Test Specification (Test Book)
**Document ID:** SVTS-COMP-002  
**Associated System:** Vehicle Control System  
**Compliance Standard:** IEC 62304:2006 + A1:2015 (Clause 5.6 & 5.7)  
**Software Safety Class:** Class C (High Criticality)  

---

## Document Control & Metadata
<table style="text-align: left; margin-left: 0; margin-right: auto; width: 100%; max-width: 600px; border-collapse: collapse;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="text-align: left; padding: 8px; border: 1px solid #dddddd; width: 30%;">Attribute</th>
      <th style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left; padding: 8px; border: 1px solid #dddddd;"><strong>Author</strong></td>
      <td style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Senior Software Engineer in Test</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 8px; border: 1px solid #dddddd;"><strong>Reviewer</strong></td>
      <td style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Principal R&D Embedded Software Engineer</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 8px; border: 1px solid #dddddd;"><strong>Approver</strong></td>
      <td style="text-align: left; padding: 8px; border: 1px solid #dddddd;">VP of Software & Robotics Quality</td>
    </tr>
    <tr>
      <td style="text-align: left; padding: 8px; border: 1px solid #dddddd;"><strong>Status</strong></td>
      <td style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Released</td>
    </tr>
  </tbody>
</table>

---

## 1. Scope & System Under Test (SUT)
This document defines the formal software integration and system verification test procedures for the Machnet Vehicle Control Software. This specification bridges the abstract behavioral logic prototyped in development notebooks with the deterministic execution validation protocols required for regulatory medical device certification.

### 1.1 Interface Configurations under Test
The software under test acts as an active communication and routing gateway between two safety-critical boundaries:
* **The High-Level Interface (UI Link):** Operates over Virtual Ethernet (TCP/IP socket protocol layer).
* **The Low-Level Interface (Robot Link):** Operates over the Vehicle CAN Bus protocol layer (simulated via byte-aligned UDP transport).

## 2. Requirements Traceability Matrix (RTM)
As mandated by IEC 62304 Clause 5.6.2, all integration test cases must maintain unidirectional traceability back to the system software requirements.

<table style="text-align: left; margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead>
    <tr>
      <th style="text-align: center; padding: 8px; border: 1px solid #dddddd;">ID</th>
      <th style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Requirement Description</th>
      <th style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Verifying Test Case ID</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align: center; padding: 8px; border: 1px solid #dddddd;">REQ-01</td><td style="padding: 8px; border: 1px solid #dddddd;">The UI shall send movement commands to the Control Software via Ethernet.</td><td style="padding: 8px; border: 1px solid #dddddd;">TC-SYS-INT-001</td></tr>
    <tr><td style="text-align: center; padding: 8px; border: 1px solid #dddddd;">REQ-02</td><td style="padding: 8px; border: 1px solid #dddddd;">The Control Software shall translate UI commands into CAN commands for the Robot.</td><td style="padding: 8px; border: 1px solid #dddddd;">TC-SYS-INT-001</td></tr>
    <tr><td style="text-align: center; padding: 8px; border: 1px solid #dddddd;">REQ-03</td><td style="padding: 8px; border: 1px solid #dddddd;">The Control Software shall monitor CAN bus for Robot sensor alerts.</td><td style="padding: 8px; border: 1px solid #dddddd;">TC-SYS-INT-002</td></tr>
    <tr><td style="text-align: center; padding: 8px; border: 1px solid #dddddd;">REQ-04</td><td style="padding: 8px; border: 1px solid #dddddd;">The Control Software shall notify the UI of critical system alerts via Ethernet.</td><td style="padding: 8px; border: 1px solid #dddddd;">TC-SYS-INT-002</td></tr>
  </tbody>
</table>

## 3. Test Environment & Infrastructure Configuration
To ensure deterministic testing independent of physical hardware availability, the execution environment leverages automated mock objects overriding the network communication boundaries.

### TC-SYS-INT-000: Test Configuration Code

In [ ]:
from unittest.mock import MagicMock

# Mocking our interfaces
mock_ethernet_socket = MagicMock() # Represents connection to UI
mock_can_bus = MagicMock()         # Represents connection to Robot

class ControlSoftware:
    def on_ethernet_message(self, msg):
        if msg == 'MOVE_FORWARD':
            mock_can_bus.send('CMD_MOTOR_FORWARD')
    
    def on_can_message(self, msg):
        if msg == 'OBSTACLE_DETECTED':
            mock_ethernet_socket.send('UI_ALERT_STOP')

control_sw = ControlSoftware()

## 4. Formal Test Case Specifications

### TC-SYS-INT-001: UI Movement Command Translation and Routing Gateway
> **Objective:** Verify that high-level abstract movement requests sent from the User Interface over the Ethernet pipe are correctly parsed, packed into byte-aligned structures, and routed as valid low-level actuation packets onto the CAN Bus network layer.

#### Preconditions / Step 01 Initialization
<table style="text-align: left; margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead><tr><th style="text-align: center; padding: 8px; border: 1px solid #dddddd;">Step</th><th style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Actions</th><th style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Expected Results</th></tr></thead>
  <tbody>
    <tr><td style="text-align: center; padding: 8px; border: 1px solid #dddddd;">01</td><td style="padding: 8px; border: 1px solid #dddddd;">Ensure the Control Software is powered on and connected to the UI and Robot interfaces.</td><td style="padding: 8px; border: 1px solid #dddddd;">System initialization successful with no interface errors. Handles evaluate to non-null descriptors.</td></tr>
  </tbody>
</table>

In [ ]:
assert mock_can_bus is not None
assert mock_ethernet_socket is not None

#### Test Execution Sequence - Command Injection
<table style="text-align: left; margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead><tr><th style="text-align: center; padding: 8px; border: 1px solid #dddddd;">Step</th><th style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Actions</th><th style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Expected Results</th></tr></thead>
  <tbody>
    <tr><td style="text-align: center; padding: 8px; border: 1px solid #dddddd;">02</td><td style="padding: 8px; border: 1px solid #dddddd;">Inject high-level Ethernet sequence representing input 'MOVE_FORWARD' into the software interface layer.</td><td style="padding: 8px; border: 1px solid #dddddd;">Control Software processes inbound socket array instantly without stalling the core event loop.</td></tr>
  </tbody>
</table>

In [ ]:
control_sw.on_ethernet_message('MOVE_FORWARD')

#### Verification Criteria - Output Route Evaluation
The Control Software must automatically catch the instruction, convert the footprint, and route 'CMD_MOTOR_FORWARD' to the Robot target location via the CAN Bus.

In [ ]:
mock_can_bus.send.assert_called_with('CMD_MOTOR_FORWARD')

### TC-SYS-INT-002: Asynchronous Robot Sensor Alert Exception Escalation
> **Objective:** Verify that safety-critical asynchronous alerts generated by the robot hardware over the low-level CAN bus link are intercepted by the control loop software instantly and translated into high-priority alert packets dispatched upstream to the User Interface over Ethernet.

#### Test Execution Sequence - Exception Injection
<table style="text-align: left; margin-left: 0; margin-right: auto; width: auto; border-collapse: collapse;">
  <thead><tr><th style="text-align: center; padding: 8px; border: 1px solid #dddddd;">Step</th><th style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Actions</th><th style="text-align: left; padding: 8px; border: 1px solid #dddddd;">Expected Results</th></tr></thead>
  <tbody>
    <tr><td style="text-align: center; padding: 8px; border: 1px solid #dddddd;">03</td><td style="padding: 8px; border: 1px solid #dddddd;">Simulate an obstacle detection event exception from the Robot sensor array into the CAN bus.</td><td style="padding: 8px; border: 1px solid #dddddd;">SUT intercepts the CAN frame interrupt flag and triggers its high-priority exception handler.</td></tr>
  </tbody>
</table>

In [ ]:
control_sw.on_can_message('OBSTACLE_DETECTED')

#### Verification Criteria - Emergency Escalation Evaluation
The Control Software must instantly notify the UI framework by transmitting 'UI_ALERT_STOP' across the Ethernet boundary layer to lock down operations.

In [ ]:
mock_ethernet_socket.send.assert_called_with('UI_ALERT_STOP')

## 5. Summary & Regression Strategy
Any modifications made to the core parsing routines, structural definitions, or interface serialization layers will mandate a full execution of this verification specification. The tests described herein must be integrated directly into the automated build verification pipeline to maintain continuous integration validation against the software system requirements.

## Test Results
**Executed at:** 2026-05-29 07:35:27

```text
==================================================================================================== test session starts ====================================================================================================
plugins: pytest-testbook-1.4.9, anyio-4.6.2.post1
git commit: 44397d07497da4eece206f9b82d7e98001873371
git branch: master
collected 6 items

✅ tests/IEC_62304_compliant_testbook.ipynb::tc-sys-int-000:_test_configuration_code **PASSED**
✅ tests/IEC_62304_compliant_testbook.ipynb::tc-sys-int-001:_ui_movement_command_translation_and_routing_gateway **PASSED**
✅ tests/IEC_62304_compliant_testbook.ipynb::tc-sys-int-001:_ui_movement_command_translation_and_routing_gateway **PASSED**
✅ tests/IEC_62304_compliant_testbook.ipynb::tc-sys-int-001:_ui_movement_command_translation_and_routing_gateway **PASSED**
✅ tests/IEC_62304_compliant_testbook.ipynb::tc-sys-int-002:_asynchronous_robot_sensor_alert_exception_escalation **PASSED**
✅ tests/IEC_62304_compliant_testbook.ipynb::tc-sys-int-002:_asynchronous_robot_sensor_alert_exception_escalation **PASSED**

==================================================================================================== 6 passed in 2.86s ====================================================================================================
```